# Módulo 1 — Segmentación K-Means
**Ciencia de Datos I · ETITC · 2026**  
Daniel Valencia · Daniel Medcalfe

Objetivo: segmentar clientes del dataset de Istanbul Mall usando K-Means.  
Criterio de validación: **Silhouette Score ≥ 0.40** y perfiles cualitativos por segmento.

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import sys
sys.path.insert(0, '../..')   # permite importar desde la raíz del proyecto

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from modules.segmentacion.kmeans import (
    preparar_features,
    calcular_elbow,
    entrenar,
    perfiles,
    guardar_resultados,
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120})

COLORES = ['#6C63FF','#43D9AD','#FFB347','#FF6584','#7EC8E3','#C7CEEA','#B5EAD7','#FFD3B6']
print("✅ Módulo kmeans importado correctamente")

## 1. Carga del dataset limpio

In [ ]:
df = pd.read_csv('../../data/processed/datos_limpios.csv')

# Normalizar nombres de columnas (compatibilidad con versión anterior del CSV)
rename_map = {'año': 'year', 'mes': 'month', 'dia_semana': 'day_of_week', 'total_compra': 'total_spend'}
df = df.rename(columns=rename_map)
df['invoice_date'] = pd.to_datetime(df['invoice_date'])

print(f"Filas    : {len(df):,}")
print(f"Columnas : {df.columns.tolist()}")
df.head(3)

## 2. Preparación de features

In [ ]:
feat_raw, feat_scaled = preparar_features(df)

print("Features utilizadas:")
print(feat_raw.drop(columns='customer_id').describe().round(2))

## 3. Método del Codo — ¿cuántos clusters elegir?

Evaluamos k = 2 … 10 con dos métricas:
- **Inercia (WCSS)**: buscamos el "codo" donde la curva se aplana
- **Silhouette Score**: buscamos el k con valor más alto (≥ 0.40 es el objetivo)

In [ ]:
print("Calculando curvas de evaluación...")
elbow_df = calcular_elbow(feat_scaled, k_max=10)
elbow_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Inercia
axes[0].plot(elbow_df['k'], elbow_df['inertia'], marker='o', color='#6C63FF', linewidth=2)
axes[0].fill_between(elbow_df['k'], elbow_df['inertia'], alpha=0.1, color='#6C63FF')
axes[0].set_title('Método del Codo — Inercia')
axes[0].set_xlabel('Número de clusters (k)')
axes[0].set_ylabel('Inercia (WCSS)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
axes[0].spines[['top','right']].set_visible(False)

# Silhouette
axes[1].plot(elbow_df['k'], elbow_df['silhouette'], marker='o', color='#43D9AD', linewidth=2)
axes[1].axhline(0.40, color='#FF6584', linestyle='--', linewidth=1.5, label='Objetivo ≥ 0.40')
axes[1].fill_between(elbow_df['k'], elbow_df['silhouette'], alpha=0.1, color='#43D9AD')
axes[1].set_title('Silhouette Score por k')
axes[1].set_xlabel('Número de clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].legend()
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('../../reports/figures/kmeans_elbow.png', bbox_inches='tight')
plt.show()
print("✅ Guardada: reports/figures/kmeans_elbow.png")

## 4. Entrenamiento del modelo final

In [ ]:
# Elegir k basándonos en el método del codo y Silhouette Score
K_OPTIMO = 4   # ajustar según las curvas de arriba

resultado, sil, inertia = entrenar(feat_raw, feat_scaled, k=K_OPTIMO)

print(f"k = {K_OPTIMO}")
print(f"Silhouette Score : {sil:.4f}  {'✅ cumple objetivo' if sil >= 0.40 else '⚠️ por debajo de 0.40'}")
print(f"Inercia          : {inertia:,.0f}")
print()
print("Distribución de clusters:")
print(resultado['cluster'].value_counts().sort_index().rename(lambda x: f'Segmento {x+1}'))

## 5. Visualización PCA — clusters en 2D

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for cl in sorted(resultado['cluster'].unique()):
    sub = resultado[resultado['cluster'] == cl]
    ax.scatter(sub['pca_x'], sub['pca_y'],
               label=f'Segmento {cl+1}  (n={len(sub):,})',
               color=COLORES[cl], alpha=0.55, s=15, linewidths=0)

ax.set_title(f'Clusters K-Means en espacio PCA 2D  (k={K_OPTIMO}, Silhouette={sil:.3f})')
ax.set_xlabel('Componente Principal 1')
ax.set_ylabel('Componente Principal 2')
ax.legend(title='Segmento', framealpha=0.7, fontsize=9)
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('../../reports/figures/kmeans_pca.png', bbox_inches='tight')
plt.show()
print("✅ Guardada: reports/figures/kmeans_pca.png")

## 6. Perfiles cualitativos por segmento

In [ ]:
tabla_perfiles = perfiles(df, resultado)
tabla_perfiles

In [ ]:
# Visualizar gasto promedio y edad por segmento
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
segmentos = [f'Seg. {i+1}' for i in tabla_perfiles.index]

# Gasto promedio
bars = axes[0].bar(segmentos, tabla_perfiles['gasto_prom'],
                   color=COLORES[:K_OPTIMO], edgecolor='none')
axes[0].bar_label(bars, labels=[f'${v:,.0f}' for v in tabla_perfiles['gasto_prom']], padding=4, fontsize=9)
axes[0].set_title('Gasto promedio por segmento')
axes[0].set_ylabel('USD')
axes[0].spines[['top','right']].set_visible(False)

# Edad promedio
bars2 = axes[1].bar(segmentos, tabla_perfiles['edad_prom'],
                    color=COLORES[:K_OPTIMO], edgecolor='none')
axes[1].bar_label(bars2, labels=[f'{v:.1f} años' for v in tabla_perfiles['edad_prom']], padding=4, fontsize=9)
axes[1].set_title('Edad promedio por segmento')
axes[1].set_ylabel('Edad')
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('../../reports/figures/kmeans_perfiles.png', bbox_inches='tight')
plt.show()
print("✅ Guardada: reports/figures/kmeans_perfiles.png")

In [ ]:
# Categoría top por segmento
df_c = df.copy()
df_c['cluster'] = resultado['cluster'].values
df_c['segmento'] = df_c['cluster'].apply(lambda x: f'Segmento {x+1}')

cat_seg = df_c.groupby(['segmento','category']).size().unstack(fill_value=0)
cat_pct = cat_seg.div(cat_seg.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(10, 5))
cat_pct.T.plot(kind='bar', ax=ax, color=COLORES[:K_OPTIMO], edgecolor='none', width=0.7)
ax.set_title('Distribución de categorías por segmento (%)')
ax.set_xlabel('Categoría')
ax.set_ylabel('% del segmento')
ax.tick_params(axis='x', rotation=35)
ax.legend(title='Segmento', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('../../reports/figures/kmeans_categorias.png', bbox_inches='tight')
plt.show()

## 7. Resumen e interpretación de segmentos

In [ ]:
print("=" * 60)
print(f"RESUMEN — K-Means con k={K_OPTIMO}")
print("=" * 60)
print(f"  Silhouette Score : {sil:.4f}  {'✅ ≥ 0.40' if sil >= 0.40 else '⚠️ < 0.40'}")
print(f"  Inercia          : {inertia:,.0f}")
print()

for i, row in tabla_perfiles.iterrows():
    print(f"  Segmento {i+1} ({row['pct_total']}% — {row['n']:,} clientes)")
    print(f"    Edad prom    : {row['edad_prom']} años")
    print(f"    Gasto prom   : ${row['gasto_prom']:,.0f}")
    print(f"    Categoría top: {row['categoria_top']}")
    print(f"    % Femenino   : {row['pct_femenino']}%")
    print(f"    Pago top     : {row['pago_top']}")
    print()

## 8. Exportar resultados

In [ ]:
guardar_resultados(
    df_original  = df,
    resultado    = resultado,
    output_path  = '../../reports/resultados/clientes_segmentados.csv'
)

print(f"\nArchivos generados en reports/:")
print("  figures/kmeans_elbow.png")
print("  figures/kmeans_pca.png")
print("  figures/kmeans_perfiles.png")
print("  figures/kmeans_categorias.png")
print("  resultados/clientes_segmentados.csv")